# Environment

In [1]:
! pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 207.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 151.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.8/347.8 kB 115.0 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
!pip install --upgrade torch transformers accelerate pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 21.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 18.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 19.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 39.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 272.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 236.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 234.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 31.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 75.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 244.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 139.0 MB/

In [3]:
! pip install huggingface_hub


[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [4]:
! pip install qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.8/34.8 MB 175.1 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


## Import Library

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import pandas as pd
import json

# Load Model

In [6]:
! nvidia-smi

Fri May 23 01:46:01 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 565.57.01              Driver Version: 565.57.01      CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:D5:00.0 Off |                    0 |
|  0%   24C    P8             29W /  300W |       1MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
# token = "HF_TOKEN_REDACTED"

In [7]:
from huggingface_hub import login
import os

token = "HF_TOKEN_REDACTED"
os.environ['HF_TOKEN'] = token

login(token=token)
print("Logged in to Hugging Face!")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face!


In [11]:
"""from huggingface_hub import login
import os

# Get HF token injected from RunPod secret
hf_token = os.getenv('HF_TOKEN')

# Login
login(token=hf_token)

print("Hugging Face login successful via RunPod Secret!")"""

'from huggingface_hub import login\nimport os\n\n# Get HF token injected from RunPod secret\nhf_token = os.getenv(\'HF_TOKEN\')\n\n# Login\nlogin(token=hf_token)\n\nprint("Hugging Face login successful via RunPod Secret!")'

## Full Model

In [8]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info         
import torch, os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

model_id = "Qwen/Qwen2.5-VL-7B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",                  
    low_cpu_mem_usage=True,
)

# limit visual tokens to keep the vision encoder light
min_pixels = 224 * 28 * 28              
max_pixels = 768 * 28 * 28              
processor = AutoProcessor.from_pretrained(
    model_id, min_pixels=min_pixels, max_pixels=max_pixels
)

/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not in

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

### Testing Response with Images

In [ ]:
# multiple images

In [ ]:
from PIL import Image
import requests
import torch
import gc

# Helper to resize images for VRAM optimization
def resize_images(image_list, size=(384, 384)):
    return [img.resize(size, Image.BILINEAR) for img in image_list]

# Load and resize images
image_urls = [
    "https://media.istockphoto.com/id/1018245126/photo/siberian-tiger-portrait.jpg?s=612x612&w=0&k=20&c=ONrnG-JrcVuqdtIjfr4Pj85ZswUPt8wD-VbJOnWYPw0=",
    "https://images.pexels.com/photos/458991/pexels-photo-458991.jpeg?cs=srgb&dl=pexels-pixabay-458991.jpg&fm=jpg"
]
original_images = [Image.open(requests.get(url, stream=True).raw).convert("RGB") for url in image_urls]
images = resize_images(original_images, size=(384, 384))


question = "What animal is shown in each image?"

messages = [
    {
        "role": "user",
        "content": [{"type": "image", "image": img} for img in images] + [{"type": "text", "text": question}]
    }
]
print(messages)
# Generate prompt with Qwen-style chat template
input_text = processor.apply_chat_template(messages, add_generation_prompt=True)

# Debug info
print("\n---- DEBUG INFO ----")
print("Generated Prompt:\n")
print(input_text)
print("\nNumber of <|vision_start|> tokens:", input_text.count("<|vision_start|>"))
print("Number of images passed:", len(images))
print("---------------------\n")

# Tokenize inputs
inputs = processor(
    text=[input_text],
    images=images,
    return_tensors="pt"
).to(model.device)

# Clear memory before generation
torch.cuda.empty_cache()
gc.collect()

# Generate response (deterministic)
with torch.amp.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu"):
    generate_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

# Decode only model's output, skipping the prompt
prompt_len = inputs["input_ids"].shape[1]
reply_ids = generate_ids[:, prompt_len:]
output_text = processor.tokenizer.decode(reply_ids[0], skip_special_tokens=True)

# Output
print("\n# Response:\n")
print(output_text)


[{'role': 'user', 'content': [{'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=384x384 at 0x7AB65A3D55D0>}, {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=384x384 at 0x7AB65A15EF50>}, {'type': 'text', 'text': 'What animal is shown in each image?'}]}]

---- DEBUG INFO ----
Generated Prompt:

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|><|vision_start|><|image_pad|><|vision_end|>What animal is shown in each image?<|im_end|>
<|im_start|>assistant


Number of <|vision_start|> tokens: 2
Number of images passed: 2
---------------------


# Response:

The first image shows a tiger, and the second image shows a cow.


# Load Datasets

In [62]:
import pandas as pd
cloud=pd.read_excel("../../Dataset/cloud/split_data/cloud_test.xlsx")
device=pd.read_excel("../../Dataset/device/split_data/device_test.xlsx")

## Cloud

In [63]:
cloud.head(2)

,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images
0,Some recently changed resources are not yet pu...,\n \nEvery once in a while i ge...,"['java', 'eclipse', 'google-app-engine', 'java...",https://stackoverflow.com/questions/49711724/s...,4,2018-04-07 20:34:12Z,1,"[""@code511788465541441, if you consider Chanse...",[{'body': '\n \nIf that happens...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png']
1,FAILED TO INITIALIZE RUN FROM PACKAGE.txt,\n \nOur pipeline indicates suc...,"['azure-web-app-service', 'azure-deployment', ...",https://stackoverflow.com/questions/65255839/f...,7,2020-12-11 17:20:22Z,2,"[""thanks for the hint. i am going to verify th...","[{'body': ""\nI am almost certain that the pack...",\nI am almost certain that the package path is...,"['https://i.sstatic.net/VJ1po.png', 'https://i..."


## Processed

In [64]:
cloud=cloud[['title','body','selected_answer','images']]

In [65]:
cloud.head(2)

,title,body,selected_answer,images
0,Some recently changed resources are not yet pu...,\n \nEvery once in a while i ge...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png']
1,FAILED TO INITIALIZE RUN FROM PACKAGE.txt,\n \nOur pipeline indicates suc...,\nI am almost certain that the package path is...,"['https://i.sstatic.net/VJ1po.png', 'https://i..."


In [66]:
cloud['context'] = "title: " + cloud['title'].astype(str) + "\nbody: " + cloud['body'].astype(str).str.strip()

In [67]:
cloud_processed=pd.DataFrame(
    {
        'context':cloud['context'],
        'gt':cloud['selected_answer'],
        'img':cloud['images']
    }
)

In [68]:
cloud_processed.head(2)

,context,gt,img
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png']
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\nI am almost certain that the package path is...,"['https://i.sstatic.net/VJ1po.png', 'https://i..."


## Device

In [69]:
device.head(2)

,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images
0,What is the purpose of the INPUT chain in the ...,\n \nWhat is the purpose of the...,"['linux', 'iptables', 'linux', 'iptables']",https://serverfault.com/questions/993878/what-...,1,2019-12-01 00:23:16Z,1,"['OK, also please add this to your answer so o...",[{'body': '\nIf some NAT operation needs to be...,\nIf some NAT operation needs to be only done ...,['https://i.sstatic.net/NUh2k.png']
1,"Logstash JDBC input, filter event timestamp di...",\n \nI am trying to change the ...,"['elasticsearch', 'logstash', 'elastic-stack',...",https://stackoverflow.com/questions/33787795/l...,0,2015-11-18 18:38:40Z,3,"['could you suggest how I could fix?', 'Your d...",[{'body': '\nOK I finally figure how to get th...,\nOK I finally figure how to get the @timestam...,"['https://i.sstatic.net/3nAZ9.png', 'https://i..."


## Processed

In [70]:
device=device[['title','body','selected_answer','images']]

In [71]:
device.head(2)

,title,body,selected_answer,images
0,What is the purpose of the INPUT chain in the ...,\n \nWhat is the purpose of the...,\nIf some NAT operation needs to be only done ...,['https://i.sstatic.net/NUh2k.png']
1,"Logstash JDBC input, filter event timestamp di...",\n \nI am trying to change the ...,\nOK I finally figure how to get the @timestam...,"['https://i.sstatic.net/3nAZ9.png', 'https://i..."


In [72]:
device['context'] = "title: " + device['title'].astype(str) + "\nbody: " + device['body'].astype(str).str.strip()

In [73]:
device_processed=pd.DataFrame(
    {
        'context':device['context'],
        'gt':device['selected_answer'],
        'img':device['images']
    }
)

In [74]:
device_processed.head(2)

,context,gt,img
0,title: What is the purpose of the INPUT chain ...,\nIf some NAT operation needs to be only done ...,['https://i.sstatic.net/NUh2k.png']
1,"title: Logstash JDBC input, filter event times...",\nOK I finally figure how to get the @timestam...,"['https://i.sstatic.net/3nAZ9.png', 'https://i..."


# Prompt Setup

## Cloud

### Zero Shot

In [75]:
zero_shot = """
You are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."

Question:
{}

Answer:
"""


In [76]:
def generate_zero_shot_prompt(row):
    return zero_shot.format(row['context'])

cloud_processed['zero-shot-question'] = cloud_processed.apply(generate_zero_shot_prompt, axis=1)

In [77]:
print(cloud_processed['zero-shot-question'][0])


You are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."

Question:
title: Some recently changed resources are not yet published
body: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I've tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?

Answer:



### CoT

In [78]:
cot = '''
You are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."
Question:
{}

Lets think step by step:
Answer:
'''

In [79]:
def generate_cot_prompt(row):
    return cot.format(row['context'])

cloud_processed['cot-question'] = cloud_processed.apply(generate_cot_prompt, axis=1)


In [80]:
print(cloud_processed['cot-question'][0])


You are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."
Question:
title: Some recently changed resources are not yet published
body: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I've tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?

Lets think step by step:
Answer:



## Device

### Zero Shot

In [81]:
zero_shot = '''
You are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."

Question:
{}

Answer:

'''

In [82]:
def generate_zero_shot_prompt(row):
    return zero_shot.format(row['context'])

device_processed['zero-shot-question'] =device_processed.apply(generate_zero_shot_prompt, axis=1)


In [83]:
print(device_processed['zero-shot-question'][0])


You are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."

Question:
title: What is the purpose of the INPUT chain in the nat table?
body: What is the purpose of the INPUT chain in the nat table ?

The PREROUTING chain in the nat table is already doing the translation of the destination addresses (DNAT) of incoming packets ...as in what is commonly called "port forwarding".

The nat:INPUT and nat:PREROUTING chains seem redundant.

What would be the typical use of the nat:INPUT chain that cannot be accomplished in the nat:PREROUTING chain ?

Answer:




### CoT

In [84]:
cot = '''
You are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."
Question:
{}

Lets think step by step:
Answer:
'''

In [85]:
def generate_cot_prompt(row):
    return cot.format(row['context'])

device_processed['cot-question'] = device_processed.apply(generate_cot_prompt, axis=1)

In [86]:
print(device_processed['cot-question'][0])


You are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.
If you lack sufficient information to answer, respond with "No answer."
Question:
title: What is the purpose of the INPUT chain in the nat table?
body: What is the purpose of the INPUT chain in the nat table ?

The PREROUTING chain in the nat table is already doing the translation of the destination addresses (DNAT) of incoming packets ...as in what is commonly called "port forwarding".

The nat:INPUT and nat:PREROUTING chains seem redundant.

What would be the typical use of the nat:INPUT chain that cannot be accomplished in the nat:PREROUTING chain ?

Lets think step by step:
Answer:



## Prompt Length

In [87]:
# extract prompt length
def prompt_length(text, add_special_tokens=True):
    if pd.isnull(text) or not isinstance(text, str) or not text.strip():
        return 0
    temp_inputs = processor.tokenizer(text, return_tensors="pt", add_special_tokens=add_special_tokens)
    return temp_inputs.input_ids.shape[1]


In [88]:
# Prompt Length -> Cloud Logs
cloud_processed['zero-shot-question-length'] = cloud_processed['zero-shot-question'].apply(prompt_length)
cloud_processed['cot-question-length'] = cloud_processed['cot-question'].apply(prompt_length)

In [89]:
# Prompt Length -> Device Logs
device_processed['zero-shot-question-length'] = device_processed['zero-shot-question'].apply(prompt_length)
device_processed['cot-question-length'] = device_processed['cot-question'].apply(prompt_length)

# Message Setup ( Prompt + Images )

In [90]:
import ast

def setup_messages(dataset, prompt_column="", image_column="img"):
    messages = []

    for _, row in dataset.iterrows():
        imgs = row.get(image_column, [])
        if isinstance(imgs, str):
            try:
                imgs = ast.literal_eval(imgs)
            except Exception:
                imgs = []

        if imgs:
            # Build structured message with image placeholders and direct question
            content = [{"type": "image", "image": img} for img in imgs if img]
            content.append({
                "type": "text",
                "text": row.get(prompt_column, "")
            })

            messages.append({
                "role": "user",
                "qa": content
            })

    return messages


In [91]:
messages = setup_messages(device_processed,prompt_column="cot-question")

In [92]:
messages[1]

{'role': 'user',
 'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/3nAZ9.png'},
  {'type': 'image', 'image': 'https://i.sstatic.net/bsoVL.png'},
  {'type': 'image', 'image': 'https://i.sstatic.net/wM2pW.png'},
  {'type': 'text',
   'text': '\nYou are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\nQuestion:\ntitle: Logstash JDBC input, filter event timestamp different to @timestamp\nbody: I am trying to change the @timestamp to match my timestamp, this is the time my event occurred at not when logstash time stamped it.\n\nhere is my conf file\n\ninput {\n jdbc {\n   jdbc_driver_library => "C:/logstash/lib/mysql-connector-java-5.1.37-bin.jar"\n   jdbc_driver_class => "com.mysql.jdbc.Driver"\n   jdbc_connection_string => "jdbc:mysql://127.0.0.1:3306/transport"\n   jdbc_user => "root"\n   jdbc_password => ""\n   schedule => "* * * * *"\n   stat

# Generate Response

## Setup Model

In [93]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)  # move model to device once
model.eval()

Qwen2_5_VLForConditionalGeneration(
  (model): Qwen2_5_VLModel(
    (visual): Qwen2_5_VisionTransformerPretrainedModel(
      (patch_embed): Qwen2_5_VisionPatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2_5_VLVisionBlock(
          (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
          (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
          (attn): Qwen2_5_VLVisionSdpaAttention(
            (qkv): Linear(in_features=1280, out_features=3840, bias=True)
            (proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): Qwen2_5_VLMLP(
            (gate_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (up_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (down_proj): Linear(in_features=3420, out_features=1280, bias=True)
            (act_fn): Si

In [94]:
import math
from PIL import Image

MAX_LONG_EDGE = 224   # longest side after down-scaling
PATCH_SIZE    = 14    # ViT patch dimension
MIN_EDGE      = 28    # shortest side must be ≥ 28 px

def img_resize(img: Image.Image) -> Image.Image:
    w, h = img.size

    # Down-scale if long edge > 224
    scale = min(1.0, MAX_LONG_EDGE / max(w, h))
    w, h = int(round(w * scale)), int(round(h * scale))

    #  Guarantee minimum size
    w, h = max(w, MIN_EDGE), max(h, MIN_EDGE)

    # nearest multiple of patch size
    w  = math.ceil(w / PATCH_SIZE) * PATCH_SIZE
    h  = math.ceil(h / PATCH_SIZE) * PATCH_SIZE

    return img.resize((w, h), Image.Resampling.LANCZOS)


In [95]:
import torch
import gc
import traceback
import requests
torch.set_grad_enabled(False)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
torch.set_grad_enabled(False)

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Load and prepare image(s) from message
def process_vision_info(messages):
    images = []
    for msg in messages:
        for item in msg.get("content", []):
            if item["type"] == "image":
                img = item["image"]
                if isinstance(img, str):
                    img = Image.open(requests.get(img, stream=True).raw).convert("RGB")
                images.append(img_resize(img))
    return images

# Process message with debug info
def process_message(msg: dict, idx: int):
    try:
        prompt_str = processor.apply_chat_template(
            [msg], tokenize=False, add_generation_prompt=True
        )

        imgs = process_vision_info([msg])
        if not imgs:
            raise ValueError("No images found in message.")

        inputs = processor(
            text=[prompt_str],
            images=imgs,
            return_tensors="pt"
        ).to(model.device)

        gen_ids = model.generate(
            **inputs,
            max_new_tokens=768,
            do_sample=False,
            use_cache=True
        )

        reply_ids = gen_ids[:, inputs.input_ids.shape[-1]:]
        answer = processor.batch_decode(
            reply_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

        return {"idx": idx, "response": answer}

    except Exception as e:
        return {
            "idx": idx,
            "response": f"[ERROR] {e}",
            "traceback": traceback.format_exc()
        }

    finally:
        for name in ("inputs", "gen_ids"):
            if name in locals():
                del locals()[name]
        torch.cuda.empty_cache()
        gc.collect()


## Testing Samples

In [43]:
#sample=cloud_processed[:5]
sample=cloud_processed[:10]

In [44]:
sample.head(2)

,context,gt,img,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png'],\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,120,126
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\nI am almost certain that the package path is...,"['https://i.sstatic.net/VJ1po.png', 'https://i...",\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,391,397


In [45]:
# Setup messages for model
sample_messages = setup_messages(sample,prompt_column="zero-shot-question")
#sample_messages = setup_messages(sample,prompt_column="cot-question")

In [46]:
sample_messages

[{'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/QQmRh.png'},
   {'type': 'text',
    'text': '\nYou are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\n\nQuestion:\ntitle: Some recently changed resources are not yet published\nbody: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I\'ve tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?\n\nAnswer:\n'}]},
 {'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/VJ1po.png'},
   {'type': 'image', 'image': 'https://i.sstatic.net/OG2ny.png'},
   {'type': 'image', 'image': 'https://i.sstatic.net/2wzYd.png'},
   {'type': 'image', 'image': 'https://i

### Get response for Sample  -> Test

In [47]:
# Output setup
sample_results = []
output_path = "sample_results_full.jsonl"

# Ensure column exists for generated responses
sample["generated_response"] = None

# Clear the output file first
with open(output_path, "w"):
    pass

# Process each message
for row_idx, msg in enumerate(tqdm(sample_messages, desc="Processing Messages")):

    # Build the message
    message = {
        "role": msg["role"],
        "content": msg["qa"]
    }

    # Get image URLs
    image_urls = [c["image"] for c in msg["qa"] if c["type"] == "image"]

    # Run inference
    result = process_message(message, idx=row_idx)
    response_text = result["response"]

    # Update DataFrame
    sample.at[row_idx, "generated_response"] = response_text

    if row_idx==0:
      print("\n#Prompt:##\n")
      print(message["content"])
      print("\n##Sample Response:##\n")
      print(response_text)

    # Save row result
    row_data = {
        "row": row_idx,
        "response": response_text,
        "image_urls": image_urls
    }
    sample_results.append(row_data)

    # Save incrementally to file
    with open(output_path, "a") as f:
        f.write(json.dumps(row_data) + "\n")


/tmp/ipykernel_645/2944900821.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample["generated_response"] = None
Processing Messages:  10%|█         | 1/10 [00:19<02:55, 19.55s/it]


#Prompt:##

[{'type': 'image', 'image': 'https://i.sstatic.net/QQmRh.png'}, {'type': 'text', 'text': '\nYou are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\n\nQuestion:\ntitle: Some recently changed resources are not yet published\nbody: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I\'ve tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?\n\nAnswer:\n'}]

##Sample Response:##

The error message indicates that some recently changed resources have not been published, which is preventing the GAE (Google App Engine) server from starting with the updated code. This can happen if the changes made to your resources are not being recognized 

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Processing Messages: 100%|██████████| 10/10 [02:51<00:00, 17.19s/it]


In [49]:
sample_results[:3]

[{'row': 0,
  'response': 'The error message indicates that some recently changed resources have not been published, which is preventing the GAE (Google App Engine) server from starting with the updated code. This can happen if the changes made to your resources are not being recognized or applied correctly by the development environment.\n\nTo resolve this issue, you might want to try the following steps:\n\n1. **Check for Pending Changes**: Ensure that all your recent changes are committed to version control. Sometimes, uncommitted changes can cause issues when trying to publish.\n\n2. **Force Publish**: In Eclipse, you can force a publish of your resources by right-clicking on the project in the Package Explorer, selecting "Team" -> "Synchronize with Team", and then choosing "Force Publish".\n\n3. **Clean and Rebuild**: Sometimes, a simple clean and rebuild can resolve the issue. Right-click on the project, select "Project" -> "Clean", and then "Project" -> "Build Project".\n\n4. **

## Cloud

In [50]:
cloud_processed.head(2)

,context,gt,img,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length
0,title: Some recently changed resources are not...,"If that happens for whatever reasons, you need...",['https://i.sstatic.net/QQmRh.png'],\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,120,126
1,title: FAILED TO INITIALIZE RUN FROM PACKAGE.t...,\nI am almost certain that the package path is...,"['https://i.sstatic.net/VJ1po.png', 'https://i...",\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,391,397


### Zero-shot

In [51]:
# Setup messages for mode
cloud_messages = setup_messages(cloud_processed,prompt_column="zero-shot-question")

In [52]:
cloud_messages[0:1]

[{'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/QQmRh.png'},
   {'type': 'text',
    'text': '\nYou are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\n\nQuestion:\ntitle: Some recently changed resources are not yet published\nbody: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I\'ve tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?\n\nAnswer:\n'}]}]

#### Get response

In [56]:
cloud_processed.loc[1131:]

,context,gt,img,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length,generated_response,cot-prompt
1130,title: Is event sourcing an enhanced pattern o...,\nThe two are compatible patterns that address...,"['https://i.sstatic.net/0hRZH.png', 'https://i...",\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,201,207,None,"{'role': 'user', 'qa': [{'type': 'image', 'ima..."
1131,title: Are Kafka Streams Appropriate for Trigg...,\n\nHow can I signal a Stream that all of the ...,['https://i.sstatic.net/CKA7t.png'],\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,569,575,None,"{'role': 'user', 'qa': [{'type': 'image', 'ima..."
1132,title: on_failure_callback triggered multiple ...,\nYou can use trigger_rule + PythonOperator to...,['https://i.sstatic.net/49U8S.png'],\nYou are an expert cloud assistant. Provide a...,\nYou are an expert cloud assistant. Provide a...,1049,1055,None,"{'role': 'user', 'qa': [{'type': 'image', 'ima..."


In [57]:
# Output setup
cloud_logs_zero_shot_results = []
output_path = "vlm_cloud_qwen2_5_vl_7b_zero_shot_results.jsonl"

# Ensure column exists for generated responses
cloud_processed["generated_response"] = None

# Clear the output file first
with open(output_path, "w"):
    pass

start_index=1131
# Process each message
for row_idx, msg in enumerate(tqdm(cloud_messages[1131:], desc="Processing Messages")):

    # Build the message
    message = {
        "role": msg["role"],
        "content": msg["qa"]
    }

    # Get image URLs
    image_urls = [c["image"] for c in msg["qa"] if c["type"] == "image"]

    # Run inference
    result = process_message(message, idx=row_idx)
    response_text = result["response"]

    # Update DataFrame
    device_processed.at[row_idx, "generated_response"] = response_text

    #Sample Response
    if row_idx==0:
      print("\n#Prompt:##\n")
      print(message["content"])
      print("\n##Sample Response:##\n")
      print(response_text)

    # Save row result
    row_data = {
        "row": row_idx+start_index,
        "response": response_text,
        "image_urls": image_urls
    }
    cloud_logs_zero_shot_results.append(row_data)

    # Save incrementally to file
    with open(output_path, "a") as f:
        f.write(json.dumps(row_data) + "\n")


Processing Messages:   0%|          | 1/491 [00:42<5:44:12, 42.15s/it]


#Prompt:##

[{'type': 'image', 'image': 'https://i.sstatic.net/CKA7t.png'}, {'type': 'text', 'text': '\nYou are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\nQuestion:\ntitle: Are Kafka Streams Appropriate for Triggering Batch Processing of Records?\nbody: Context\nI have three services in place, each of which generate a certain JSON payload (and take different times to do so) that is needed to be able to process a message which is the result of combining all three JSON payloads into a single payload. This final payload in turn is to be sent to another Kafka Topic so that it can then be consumed by another service.\nBelow you can find a diagram that better explains the problem at hand. The information aggregator service receives a request to aggregate information, it sends that request to a Kafka topic so that Service 1, Service 2 and Service 3 co

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Processing Messages:  55%|█████▍    | 270/491 [1:33:04<1:07:08, 18.23s/it]/usr/local/lib/python3.10/dist-packages/PIL/Image.py:1043: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Processing Messages: 100%|██████████| 491/491 [2:49:59<00:00, 20.77s/it]


In [ ]:
cloud_processed['zero-shot-prompt'] = setup_messages(cloud_processed, prompt_column="zero-shot-question")

In [ ]:
cloud_processed.to_excel("../responses/cloud/vlm_response_cloud_qwen_7b_zero_shot.xlsx", index=False)

### CoT

In [43]:
# Setup messages for mode
cloud_messages = setup_messages(cloud_processed,prompt_column="cot-question")

In [ ]:
cloud_messages[0:1]

[{'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/QQmRh.png'},
   {'type': 'text',
    'text': '\nYou are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\nQuestion:\ntitle: Some recently changed resources are not yet published\nbody: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I\'ve tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?\n\nLets think step by step:\nAnswer:\n'}]}]

#### Get response

In [ ]:
# Output setup
cloud_logs_cot_results = []
output_path = "vlm_cloud_qwen2_5_vl_7b_cot_results.jsonl"

# Ensure column exists for generated responses
cloud_processed["generated_response"] = None

# Clear the output file first
with open(output_path, "w"):
    pass

# Process each message
for row_idx, msg in enumerate(tqdm(cloud_messages, desc="Processing Messages")):

    # Build the message
    message = {
        "role": msg["role"],
        "content": msg["qa"]
    }

    # Get image URLs
    image_urls = [c["image"] for c in msg["qa"] if c["type"] == "image"]

    # Run inference
    result = process_message(message, idx=row_idx)
    response_text = result["response"]

    # Update DataFrame
    device_processed.at[row_idx, "generated_response"] = response_text

    #Sample Response
    if row_idx==0:
      print("\n#Prompt:##\n")
      print(message["content"])
      print("\n##Sample Response:##\n")
      print(response_text)

    # Save row result
    row_data = {
        "row": row_idx,
        "response": response_text,
        "image_urls": image_urls
    }
    cloud_logs_cot_results.append(row_data)

    # Save incrementally to file
    with open(output_path, "a") as f:
        f.write(json.dumps(row_data) + "\n")


Processing Messages:   0%|          | 1/1622 [00:19<8:41:03, 19.29s/it]


#Prompt:##

[{'type': 'image', 'image': 'https://i.sstatic.net/QQmRh.png'}, {'type': 'text', 'text': '\nYou are an expert cloud assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\nQuestion:\ntitle: Some recently changed resources are not yet published\nbody: Every once in a while i get this error message. If I click Yes then my GAE server starts but with old code. I\'ve tried cleaning, restarting eclipse, shutting down computer, closing and opening the project again, deleting the launch config and adding it again but nothing works. Does anyone know what causes this issue?\n\nLets think step by step:\nAnswer:\n'}]

##Sample Response:##

The error message indicates that some recently changed resources have not been published, which is causing the issue. When you click "Yes," your GAE server starts but with old code because the changes have not been deployed.

To resolve 

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Processing Messages:   7%|▋         | 112/1622 [32:28<6:05:43, 14.53s/it]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [46]:
cloud_processed['cot-prompt'] = setup_messages(cloud_processed, prompt_column="cot-question")

In [47]:
cloud_processed.to_excel("../responses/cloud/vlm_response_cloud_qwen_7b.xlsx", index=False)

## Device

In [96]:
device_processed.head(2)

,context,gt,img,zero-shot-question,cot-question,zero-shot-question-length,cot-question-length
0,title: What is the purpose of the INPUT chain ...,\nIf some NAT operation needs to be only done ...,['https://i.sstatic.net/NUh2k.png'],\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,150,156
1,"title: Logstash JDBC input, filter event times...",\nOK I finally figure how to get the @timestam...,"['https://i.sstatic.net/3nAZ9.png', 'https://i...",\nYou are an expert device assistant. Provide ...,\nYou are an expert device assistant. Provide ...,586,592


### Zero-Shot

In [97]:
# Setup messages for mode
device_messages = setup_messages(device_processed,prompt_column="zero-shot-question")

In [98]:
device_messages[:2]

[{'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/NUh2k.png'},
   {'type': 'text',
    'text': '\nYou are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\n\nQuestion:\ntitle: What is the purpose of the INPUT chain in the nat table?\nbody: What is the purpose of the INPUT chain in the nat table ?\n\nThe PREROUTING chain in the nat table is already doing the translation of the destination addresses (DNAT) of incoming packets ...as in what is commonly called "port forwarding".\n\nThe nat:INPUT and nat:PREROUTING chains seem redundant.\n\nWhat would be the typical use of the nat:INPUT chain that cannot be accomplished in the nat:PREROUTING chain ?\n\nAnswer:\n\n'}]},
 {'role': 'user',
  'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/3nAZ9.png'},
   {'type': 'image', 'image': 'https://i.sstatic.net/bsoVL.png'},
   {'ty

#### Get response

In [99]:
# Output setup
device_logs_zero_shot_results = []
output_path = "vlm_device_qwen2_5_vl_7b_zero_shot_results.jsonl"

# Ensure column exists for generated responses
device_processed["generated_response"] = None

# Clear the output file first
with open(output_path, "w"):
    pass

# Process each message
for row_idx, msg in enumerate(tqdm(device_messages, desc="Processing Messages")):

    # Build the message
    message = {
        "role": msg["role"],
        "content": msg["qa"]
    }

    # Get image URLs
    image_urls = [c["image"] for c in msg["qa"] if c["type"] == "image"]

    # Run inference
    result = process_message(message, idx=row_idx)
    response_text = result["response"]

    # Update DataFrame
    device_processed.at[row_idx, "generated_response"] = response_text

    #Sample Response
    if row_idx==0:
      print("\n#Prompt:##\n")
      print(message["content"])
      print("\n##Sample Response:##\n")
      print(response_text)


    # Save row result
    row_data = {
        "row": row_idx,
        "response": response_text,
        "image_urls": image_urls
    }
    device_logs_zero_shot_results.append(row_data)

    # Save incrementally to file
    with open(output_path, "a") as f:
        f.write(json.dumps(row_data) + "\n")


Processing Messages:   0%|          | 1/993 [00:05<1:32:55,  5.62s/it]


#Prompt:##

[{'type': 'image', 'image': 'https://i.sstatic.net/NUh2k.png'}, {'type': 'text', 'text': '\nYou are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\n\nQuestion:\ntitle: What is the purpose of the INPUT chain in the nat table?\nbody: What is the purpose of the INPUT chain in the nat table ?\n\nThe PREROUTING chain in the nat table is already doing the translation of the destination addresses (DNAT) of incoming packets ...as in what is commonly called "port forwarding".\n\nThe nat:INPUT and nat:PREROUTING chains seem redundant.\n\nWhat would be the typical use of the nat:INPUT chain that cannot be accomplished in the nat:PREROUTING chain ?\n\nAnswer:\n\n'}]

##Sample Response:##

The nat:INPUT chain in the nat table is typically used for network address translation (NAT) on packets that have already been routed to the host but have not yet

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Processing Messages:  16%|█▋        | 162/993 [37:43<2:09:46,  9.37s/it]/usr/local/lib/python3.10/dist-packages/PIL/Image.py:1043: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Processing Messages: 100%|██████████| 993/993 [4:05:18<00:00, 14.82s/it]


In [100]:
device_messages[1]

{'role': 'user',
 'qa': [{'type': 'image', 'image': 'https://i.sstatic.net/3nAZ9.png'},
  {'type': 'image', 'image': 'https://i.sstatic.net/bsoVL.png'},
  {'type': 'image', 'image': 'https://i.sstatic.net/wM2pW.png'},
  {'type': 'text',
   'text': '\nYou are an expert device assistant. Provide a concise answer to the following question, and use the images as reference.\nIf you lack sufficient information to answer, respond with "No answer."\n\nQuestion:\ntitle: Logstash JDBC input, filter event timestamp different to @timestamp\nbody: I am trying to change the @timestamp to match my timestamp, this is the time my event occurred at not when logstash time stamped it.\n\nhere is my conf file\n\ninput {\n jdbc {\n   jdbc_driver_library => "C:/logstash/lib/mysql-connector-java-5.1.37-bin.jar"\n   jdbc_driver_class => "com.mysql.jdbc.Driver"\n   jdbc_connection_string => "jdbc:mysql://127.0.0.1:3306/transport"\n   jdbc_user => "root"\n   jdbc_password => ""\n   schedule => "* * * * *"\n   st

In [ ]:
device_logs_zero_shot_results

In [101]:
device_processed['zero-shot-prompt'] = setup_messages(device_processed, prompt_column="zero-shot-question")


In [102]:
device_processed.to_excel("../responses/device/vlm_response_device_qwen_7b_zero_shot.xlsx", index=False)

### CoT

In [ ]:
# Setup messages for mode
device_messages = setup_messages(device_processed,prompt_column="cot-question")

In [ ]:
device_messages[:2]

#### Get response

In [ ]:
# Output setup
device_logs_cot_results = []
output_path = "vlm_device_qwen2_5_vl_7b_cot_results.jsonl"

# Ensure column exists for generated responses
device_processed["generated_response"] = None

# Clear the output file first
with open(output_path, "w"):
    pass

# Process each message
for row_idx, msg in enumerate(tqdm(device_messages, desc="Processing Messages")):

    # Build the message
    message = {
        "role": msg["role"],
        "content": msg["qa"]
    }

    # Get image URLs
    image_urls = [c["image"] for c in msg["qa"] if c["type"] == "image"]

    # Run inference
    result = process_message(message, idx=row_idx)
    response_text = result["response"]

    # Update DataFrame
    device_processed.at[row_idx, "generated_response"] = response_text

    #Sample Response
    if row_idx==0:
      print("\n#Prompt:##\n")
      print(message["content"])
      print("\n##Sample Response:##\n")
      print(response_text)


    # Save row result
    row_data = {
        "row": row_idx,
        "response": response_text,
        "image_urls": image_urls
    }
    device_logs_cot_results.append(row_data)

    # Save incrementally to file
    with open(output_path, "a") as f:
        f.write(json.dumps(row_data) + "\n")


In [ ]:
device_logs_cot_results

In [ ]:
device_processed['cot-prompt'] = setup_messages(cloud_processed, prompt_column="cot-question")


In [ ]:
device_processed.to_excel("../responses/device/vlm_response_device_qwen_7b.xlsx", index=False)